# 70. Serving Scheduler Benchmark | 推理服务调度基准
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `推理服务`, `调度`, `基准对比` | **目标人群：** 项目决策练习者

---

## 本节导读

本节从一组可复现的请求流和资源配置开始，先建立 serving baseline，再分别观察负载变化与调度策略对请求等待和服务能力的影响。
学习过程中，你会把“整体吞吐变高”与“部分请求等待更久”放在同一张结果表里，最后根据延迟、服务能力和公平性判断方案适合什么负载。

**关键词：** `serving scheduler`, `TTFT`, `TPOT`, `throughput`, `fairness`, `utilization`

---

## 前置阅读

**导语：** 开始前先了解 decode 调度、KV Cache 调度、PD 分离和前缀缓存如何改变请求的等待与执行顺序。阅读时关注请求状态、资源分配和服务指标之间的联系，随后把这些机制放入统一的 serving workload 中观察。
- [36. Decode Scheduling | 解码调度](./36_Decode_Scheduling.ipynb)
- [37. KV Cache Scheduling | KV Cache 调度](./37_KV_Cache_Scheduling.ipynb)
- [38. Prefill/Decode Scheduling | Prefill/Decode 调度](./38_Prefill_Decode_Scheduling.ipynb)

---

### Step 1：建立请求状态与调度全景

一个 serving 调度器持续处理同一条状态链：请求到达后进入 pending，被选入 batch 后进入 active，完成后进入 completed。调度策略决定谁先进入 batch，批次组成决定谁共享一次执行窗口，完成结果又决定队列中其余请求的等待。

| 状态或动作 | 发生了什么 | 会影响什么 |
|:---|:---|:---|
| pending | 请求已到达、尚未获准执行 | queue wait、队列长度 |
| 选择策略 | FIFO、shortest 或 priority 从 ready queue 选择请求 | 完成顺序与公平性 |
| active batch | 至多 batch_size 个请求共享一次服务窗口 | 批次耗时、utilization、吞吐 |
| completed | batch 结束后请求完成并从队列移除 | E2E、完成数与下一轮等待 |
| 策略对照 | 对同一请求流比较 baseline 与 candidate | 是否接受、继续调优或拒绝 |

![Serving 实例级调度：请求到达、排队、Batch 组织与策略决策](../docs/public/02_PyTorch_Algorithms/70_serving_scheduler_scope.svg)


### Step 2：从请求状态到批次准入

CPU 模型固定请求流、单 worker 与服务成本，只改变调度策略或 batch 参数。每轮只从已经到达的 pending 请求中选择一批；未选中的请求继续留在队列，已完成的请求不能再次进入后续 batch。每条请求至少包含 `request_id`、`arrival_ms`、`service_ms` 和 `output_tokens`；`priority` 只在优先级策略中参与排序。

| 对照对象 | 固定条件 | 唯一变化 | 必须保持的不变量 |
|:---|:---|:---|:---|
| FIFO baseline | 请求流、单 worker、服务成本 | FIFO，batch_size=1 | 每个请求只完成一次 |
| 策略对照 | 同一请求流与 batch 上限 | FIFO / shortest / priority | 未到达请求不能被提前选择 |
| 批次对照 | 同一请求流与选择策略 | batch_size 或 batch_speedup | batch 不超过上限；未选请求留在队列 |
| 到达模式对照 | 同一服务成本与策略 | arrival_ms | 只有已到达请求可参与当前轮 |



### Step 3：用等待、吞吐与公平性解释策略取舍

先读取每轮 batch trace，确认策略确实改变了谁在何时进入服务窗口；再汇总请求级和服务级指标。吞吐提高并不自动代表更好的交互体验，长尾等待与公平性必须同时进入判断。

| 指标 | 单位 | 解释时要结合什么 |
|:---|:---|:---|
| queue_wait_ms / ttft_ms | ms | 请求到达至被 batch 接纳的等待；CPU 模型用它近似首 token 前的排队时间 |
| makespan_ms | ms | 整组请求结束时间，结合 batch trace 判断批处理是否缩短总时间 |
| throughput_tps | token/s | 总输出 token 除以 makespan，不能脱离等待时间单独判断 |
| utilization | 0–1 | worker 的忙碌占比；高利用率若伴随高等待并非直接收益 |
| fairness | 0–1 | 请求等待差距的教学代理；与完成顺序一起判断是否牺牲部分请求 |
| completed_count | 请求数 | 必须等于输入请求数，且每个 request_id 只能完成一次 |


### Step 4：实现 CPU 调度机制与决策

题目区把 serving 调度收成三个可测试的机制动作：选择已到达请求、推进一个 batch 的状态与时间、根据 baseline/candidate 的多指标对照形成决策。完成这些动作后，骨架会汇总等待、吞吐和决策字段。

| TODO | 函数 | 机制责任 | 关键测试 |
|:---|:---|:---|:---|
| TODO 1 | select_next_batch | 按 FIFO、shortest 或 priority 从 ready queue 选择有限 batch | 稳定顺序、优先级、batch 上限、剩余队列 |
| TODO 2 | simulate_serving_scheduler | 用批内最长服务时间推进 pending 到 active 到 completed | 不提前执行、请求只完成一次、batch trace 与等待一致 |
| TODO 3 | recommend_serving_scheduler_run | 结合延迟、吞吐、utilization 与公平性输出决策 | accept / tune / reject 的多指标条件 |
| 辅助函数 | summarize 和 compare | 汇总重复运行并统一 candidate 减 baseline 方向 | 空输入、字段契约与 delta 方向 |


In [ ]:
from typing import Dict, List

In [ ]:
# 题目区只挖空三项调度机制：批次选择、状态推进与多指标决策。
# 请求字段检查、状态记录、汇总指标和报告字段由骨架提供。

def select_next_batch(queue: List[Dict[str, float]], policy: str, batch_size: int) -> tuple[list[Dict[str, float]], list[Dict[str, float]]]:
    """从已到达的 pending 队列中选择一个不超过上限的 active batch。

    ``shortest`` 按 ``service_ms`` 排序，``priority`` 优先选择较大 priority，
    ``fifo`` 保持到达与输入顺序。返回值分别是本轮 active batch 与剩余 pending 队列。
    """
    if policy not in {'fifo', 'shortest', 'priority'}:
        raise ValueError('policy 只能是 fifo、shortest 或 priority')
    if batch_size < 1:
        raise ValueError('batch_size 必须 >= 1')
    # TODO 1（批次选择）：按 policy 排序后选择最多 batch_size 个已到达请求。
    # ordered = ???  # shortest: service_ms；priority: -priority；再以 arrival_ms、_index 稳定排序。
    # batch = ???    # 取前 batch_size 个；其余请求必须保留在 pending 队列。
    raise NotImplementedError('TODO 1：请完成请求选择机制')

# TODO 2：推进请求状态、批处理和完成时间
def simulate_serving_scheduler(requests: List[Dict[str, float]], policy: str = 'fifo', batch_size: int = 1, batch_speedup: float = 1.0) -> Dict[str, object]:
    """用单 worker CPU 模型推进 pending、active 与 completed 三种状态。

    请求到达后才可被选择；每轮至多选择 ``batch_size`` 个请求。批内最慢请求决定
    服务窗口，``batch_speedup`` 表示教学模型中的批处理收益；``queue_wait_ms`` 是
    首 token 前排队时间的代理，``batch_trace`` 记录每个 active 窗口。
    """
    if policy not in {'fifo', 'shortest', 'priority'}:
        raise ValueError('policy 只能是 fifo、shortest 或 priority')
    if batch_size < 1 or batch_speedup <= 0:
        raise ValueError('batch_size 必须 >= 1，batch_speedup 必须 > 0')
    if not isinstance(requests, list):
        raise TypeError('requests 必须是 list[dict]')
    if not requests:
        return {'request_count': 0, 'completed_count': 0, 'makespan_ms': 0.0, 'throughput_tps': 0.0, 'utilization': 0.0, 'fairness': 1.0, 'requests': []}
    pending = []
    for index, item in enumerate(requests):
        arrival = float(item.get('arrival_ms', 0.0))
        service = float(item.get('service_ms', 0.0))
        tokens = int(item.get('output_tokens', 0))
        if arrival < 0 or service <= 0 or tokens < 0:
            raise ValueError('arrival_ms >= 0，service_ms > 0，output_tokens >= 0')
        pending.append({**item, '_index': index, 'arrival_ms': arrival, 'service_ms': service, 'output_tokens': tokens})
    pending.sort(key=lambda item: (item['arrival_ms'], item['_index']))
    queue, completed, batch_trace, cursor = [], [], [], 0
    now = busy = 0.0
    while cursor < len(pending) or queue:
        if not queue and cursor < len(pending):
            now = max(now, pending[cursor]['arrival_ms'])
        while cursor < len(pending) and pending[cursor]['arrival_ms'] <= now:
            queue.append(pending[cursor]); cursor += 1
        batch, queue = select_next_batch(queue, policy, batch_size)
        start = now
        # TODO 2（状态推进）：active batch 由最慢请求决定服务窗口，再由 batch_speedup 摊薄。
        # duration = ???
        raise NotImplementedError('TODO 2：请完成 batch 服务时间模型')
        batch_trace.append({'start_ms': round(start, 4), 'duration_ms': round(duration, 4), 'request_ids': [item.get('request_id', f"request-{item['_index']}") for item in batch]})
        now += duration; busy += duration
        for item in batch:
            completed.append({
                'request_id': item.get('request_id', f"request-{item['_index']}"),
                'queue_wait_ms': round(start - item['arrival_ms'], 4),
                'ttft_ms': round(start - item['arrival_ms'], 4),
                'e2e_ms': round(now - item['arrival_ms'], 4),
                'output_tokens': item['output_tokens'],
            })
    makespan = now - min(item['arrival_ms'] for item in pending)
    waits = [item['ttft_ms'] for item in completed]
    mean_wait = sum(waits) / len(waits) if waits else 0.0
    spread = (max(waits) - min(waits)) if waits else 0.0
    fairness = max(0.0, 1.0 - spread / max(mean_wait, 1.0))
    total_tokens = sum(item['output_tokens'] for item in completed)
    avg_queue_wait_ms = sum(item['queue_wait_ms'] for item in completed) / len(completed) if completed else 0.0
    return {
        'request_count': len(pending), 'completed_count': len(completed),
        'makespan_ms': round(makespan, 4),
        'throughput_tps': round(total_tokens / (makespan / 1000.0), 4) if makespan else 0.0,
        'utilization': round(busy / makespan, 4) if makespan else 0.0,
        'avg_queue_wait_ms': round(avg_queue_wait_ms, 4),
        'fairness': round(fairness, 4), 'batch_trace': batch_trace, 'requests': completed,
    }

# 辅助函数：汇总调度运行结果
def summarize_serving_scheduler_runs(runs: List[Dict[str, float]]) -> Dict[str, object]:
    """汇总同一请求流和 batch 配置下的调度运行结果。"""
    if not isinstance(runs, list):
        raise TypeError('runs 必须是 list[dict]')
    required = {'name', 'ttft_ms', 'throughput_tps'}
    for index, item in enumerate(runs):
        if not isinstance(item, dict) or not required.issubset(item):
            raise ValueError(f'第 {index} 条 run 必须包含 {sorted(required)}')
    run_count = len(runs)
    avg_throughput_tps = sum(item['throughput_tps'] for item in runs) / run_count if run_count else 0.0
    best = min(runs, key=lambda item: item['ttft_ms']) if runs else None
    return {'run_count': run_count, 'best_latency_run': best.get('name', 'run') if best else None, 'avg_throughput_tps': avg_throughput_tps}

# 辅助函数：比较 baseline 与 candidate
def compare_scheduler_to_baseline(baseline: Dict[str, float], candidate: Dict[str, float]) -> Dict[str, float]:
    """计算 candidate 相对 baseline 的延迟、吞吐、公平性和利用率变化。"""
    required = {'ttft_ms', 'tpot_ms', 'throughput_tps', 'fairness', 'utilization'}
    for name, item in (('baseline', baseline), ('candidate', candidate)):
        if not isinstance(item, dict) or not required.issubset(item):
            raise ValueError(f'{name} 必须包含 {sorted(required)}')
    delta_specs = [
        ('ttft_ms', 'ttft_delta_ms'),
        ('tpot_ms', 'tpot_delta_ms'),
        ('throughput_tps', 'throughput_gain_tps'),
        ('fairness', 'fairness_delta'),
        ('utilization', 'utilization_delta'),
    ]
    comparison = {
        output_key: round(candidate[source_key] - baseline[source_key], 4)
        for source_key, output_key in delta_specs
    }
    return comparison

# TODO 3：输出 serving 调度决策
def recommend_serving_scheduler_run(
    baseline: Dict[str, float], candidate: Dict[str, float], min_throughput_gain: float, min_fairness: float
) -> Dict[str, object]:
    """Choose a scheduler action from latency, throughput, utilization and fairness.

    min_throughput_gain is candidate minus baseline token/s; min_fairness is an
    absolute candidate fairness floor. Accept requires all user-facing metrics
    to pass, while tune retains a candidate with partial capacity improvement.
    """
    if min_throughput_gain < 0:
        raise ValueError('min_throughput_gain 不能为负数')
    if not 0 <= min_fairness <= 1:
        raise ValueError('min_fairness 必须位于 [0, 1]')
    comparison = compare_scheduler_to_baseline(baseline, candidate)
    throughput_ok = comparison['throughput_gain_tps'] >= min_throughput_gain
    fairness_ok = candidate['fairness'] >= min_fairness
    latency_ok = comparison['ttft_delta_ms'] < 0 and comparison['tpot_delta_ms'] <= 0
    # TODO 3（多指标决策）：交互延迟、服务能力与公平性必须一起通过才能 accept。
    if latency_ok and throughput_ok and fairness_ok:
        return {'decision': 'accept', 'reason': '延迟、吞吐和公平性都达标，适合进入真实 serving 验证', 'next_action': 'promote_to_serving_rollout'}
    if comparison['throughput_gain_tps'] >= 0 and comparison['utilization_delta'] >= 0:
        return {'decision': 'tune', 'reason': '吞吐和利用率已有改善，但公平性或延迟边界还不够稳', 'next_action': 'refine_queue_rules_or_worker_split'}
    return {'decision': 'reject', 'reason': 'candidate 没有形成可信的 serving 调度收益', 'next_action': 'fallback_to_scheduler_audit'}


In [ ]:
# 测试你的实现
# 测试目标：分别验证队列状态转移、运行汇总、baseline 对照和 accept/tune/reject 决策。
# 这些断言检查机制契约与指标方向，不把 CPU 教学耗时当作真实 GPU serving 性能。
REQUESTS = [
    {'request_id': 'r1', 'arrival_ms': 0, 'service_ms': 10, 'output_tokens': 20, 'priority': 1},
    {'request_id': 'r2', 'arrival_ms': 0, 'service_ms': 2, 'output_tokens': 20, 'priority': 3},
    {'request_id': 'r3', 'arrival_ms': 1, 'service_ms': 3, 'output_tokens': 20, 'priority': 2},
]

BASELINE = {
    'name': 'fifo',
    'ttft_ms': 220,
    'tpot_ms': 42,
    'throughput_tps': 180,
    'fairness': 0.78,
    'utilization': 0.70,
}

CANDIDATE = {
    'name': 'priority_scheduler',
    'ttft_ms': 180,
    'tpot_ms': 36,
    'throughput_tps': 205,
    'fairness': 0.81,
    'utilization': 0.79,
}


def _assert_value_error(callable_, message):
    try:
        callable_()
    except ValueError:
        return
    raise AssertionError(message)


def test_serving_simulation_contract():
    """验证请求完成一次、等待时间和聚合指标之间的契约。"""
    simulation = simulate_serving_scheduler(REQUESTS, policy='fifo', batch_size=1)
    assert simulation['request_count'] == 3
    assert simulation['completed_count'] == 3
    assert simulation['makespan_ms'] == 15.0
    assert simulation['throughput_tps'] == 4000.0
    assert simulation['utilization'] == 1.0
    assert simulation['avg_queue_wait_ms'] == round((0.0 + 10.0 + 11.0) / 3, 4)
    assert simulation['requests'][0]['queue_wait_ms'] == 0.0
    assert simulation['requests'][1]['ttft_ms'] == 10.0
    assert [trace['request_ids'] for trace in simulation['batch_trace']] == [['r1'], ['r2'], ['r3']]
    assert 0.0 <= simulation['fairness'] <= 1.0


def test_policy_batching_and_input_guards():
    """验证 FIFO/shortest、batch 边界、空输入和非法配置。"""
    shortest = simulate_serving_scheduler(REQUESTS, policy='shortest', batch_size=1)
    assert shortest['requests'][0]['request_id'] == 'r2'
    assert shortest['requests'][-1]['request_id'] == 'r1'
    priority = simulate_serving_scheduler(REQUESTS, policy='priority', batch_size=1)
    assert priority['requests'][0]['request_id'] == 'r2'
    assert priority['batch_trace'][0]['request_ids'] == ['r2']
    batched = simulate_serving_scheduler(REQUESTS, policy='fifo', batch_size=2, batch_speedup=1.0)
    assert batched['completed_count'] == len(REQUESTS)
    assert batched['makespan_ms'] == 13.0
    assert batched['makespan_ms'] < 15.0
    empty = simulate_serving_scheduler([])
    assert empty['request_count'] == 0 and empty['completed_count'] == 0
    _assert_value_error(lambda: simulate_serving_scheduler(REQUESTS, policy='unknown'), '非法调度策略应明确拒绝！')
    for invalid in ({'batch_size': 0}, {'batch_speedup': 0.0}):
        _assert_value_error(lambda invalid=invalid: simulate_serving_scheduler(REQUESTS, **invalid), '非法 batch 参数应明确拒绝！')


def test_result_summary_contract():
    """验证同一 workload 的运行汇总和最低 TTFT 选择。"""
    summary = summarize_serving_scheduler_runs([BASELINE, CANDIDATE])
    assert summary['run_count'] == 2
    assert summary['best_latency_run'] == 'priority_scheduler'
    assert summary['avg_throughput_tps'] == 192.5
    assert summarize_serving_scheduler_runs([]) == {
        'run_count': 0,
        'best_latency_run': None,
        'avg_throughput_tps': 0.0,
    }


def test_baseline_comparison_contract():
    """验证 candidate-baseline 的差值方向和指标解释。"""
    comparison = compare_scheduler_to_baseline(BASELINE, CANDIDATE)
    assert comparison['ttft_delta_ms'] == -40
    assert comparison['tpot_delta_ms'] == -6
    assert comparison['throughput_gain_tps'] == 25
    assert comparison['fairness_delta'] == 0.03
    assert comparison['utilization_delta'] == 0.09
    assert comparison['throughput_gain_tps'] > 0
    assert comparison['fairness_delta'] > 0


def test_decision_policy_contract():
    """验证同时达标、部分改善和无收益三种决策。"""
    decision = recommend_serving_scheduler_run(
        BASELINE,
        CANDIDATE,
        min_throughput_gain=20,
        min_fairness=0.8,
    )
    assert decision['decision'] == 'accept'
    assert decision['next_action'] == 'promote_to_serving_rollout'

    weak_candidate = {
        'name': 'aggressive_batching',
        'ttft_ms': 195,
        'tpot_ms': 38,
        'throughput_tps': 202,
        'fairness': 0.76,
        'utilization': 0.82,
    }
    weak_decision = recommend_serving_scheduler_run(
        BASELINE,
        weak_candidate,
        min_throughput_gain=20,
        min_fairness=0.8,
    )
    assert weak_decision['decision'] == 'tune'

    bad_candidate = {
        'name': 'overfit_scheduler',
        'ttft_ms': 260,
        'tpot_ms': 48,
        'throughput_tps': 170,
        'fairness': 0.60,
        'utilization': 0.66,
    }
    bad_decision = recommend_serving_scheduler_run(
        BASELINE,
        bad_candidate,
        min_throughput_gain=20,
        min_fairness=0.8,
    )
    assert bad_decision['decision'] == 'reject'


def test_serving_scheduler_benchmark_template():
    test_serving_simulation_contract()
    test_policy_batching_and_input_guards()
    test_result_summary_contract()
    test_baseline_comparison_contract()
    test_decision_policy_contract()
    print('测试通过：推理服务调度基准模板可以工作。')


test_serving_scheduler_benchmark_template()


## 参考代码与解析

### 代码

In [ ]:
from typing import Dict, List
def select_next_batch(queue: List[Dict[str, float]], policy: str, batch_size: int) -> tuple[list[Dict[str, float]], list[Dict[str, float]]]:
    """从已到达的 pending 队列中选择一个不超过上限的 active batch。

    ``shortest`` 按 ``service_ms`` 排序，``priority`` 优先选择较大 priority，
    ``fifo`` 保持到达与输入顺序。返回值分别是本轮 active batch 与剩余 pending 队列。
    """
    if policy not in {'fifo', 'shortest', 'priority'}:
        raise ValueError('policy 只能是 fifo、shortest 或 priority')
    if batch_size < 1:
        raise ValueError('batch_size 必须 >= 1')
    # TODO 1：稳定排序后切分 batch；剩余请求继续保持 pending。
    if policy == 'shortest':
        ordered = sorted(queue, key=lambda item: (item['service_ms'], item['_index']))
    elif policy == 'priority':
        ordered = sorted(queue, key=lambda item: (-item.get('priority', 0.0), item['arrival_ms'], item['_index']))
    else:
        ordered = list(queue)
    return ordered[:batch_size], ordered[batch_size:]

# TODO 2：推进请求状态、批处理和完成时间
def simulate_serving_scheduler(requests: List[Dict[str, float]], policy: str = 'fifo', batch_size: int = 1, batch_speedup: float = 1.0) -> Dict[str, object]:
    """用单 worker CPU 模型推进 pending、active 与 completed 三种状态。

    请求到达后才可被选择；每轮至多选择 ``batch_size`` 个请求。批内最慢请求决定
    服务窗口，``batch_speedup`` 表示教学模型中的批处理收益；``queue_wait_ms`` 是
    首 token 前排队时间的代理，``batch_trace`` 记录每个 active 窗口。
    """
    if policy not in {'fifo', 'shortest', 'priority'}:
        raise ValueError('policy 只能是 fifo、shortest 或 priority')
    if batch_size < 1 or batch_speedup <= 0:
        raise ValueError('batch_size 必须 >= 1，batch_speedup 必须 > 0')
    if not isinstance(requests, list):
        raise TypeError('requests 必须是 list[dict]')
    if not requests:
        return {'request_count': 0, 'completed_count': 0, 'makespan_ms': 0.0, 'throughput_tps': 0.0, 'utilization': 0.0, 'fairness': 1.0, 'requests': []}
    pending = []
    for index, item in enumerate(requests):
        arrival = float(item.get('arrival_ms', 0.0))
        service = float(item.get('service_ms', 0.0))
        tokens = int(item.get('output_tokens', 0))
        if arrival < 0 or service <= 0 or tokens < 0:
            raise ValueError('arrival_ms >= 0，service_ms > 0，output_tokens >= 0')
        pending.append({**item, '_index': index, 'arrival_ms': arrival, 'service_ms': service, 'output_tokens': tokens})
    pending.sort(key=lambda item: (item['arrival_ms'], item['_index']))
    queue, completed, batch_trace, cursor = [], [], [], 0
    now = busy = 0.0
    while cursor < len(pending) or queue:
        if not queue and cursor < len(pending):
            now = max(now, pending[cursor]['arrival_ms'])
        while cursor < len(pending) and pending[cursor]['arrival_ms'] <= now:
            queue.append(pending[cursor]); cursor += 1
        batch, queue = select_next_batch(queue, policy, batch_size)
        start = now
        # TODO 2：同一 active batch 的服务窗口由最慢请求决定，再应用 batch_speedup。
        duration = max(item['service_ms'] for item in batch) / batch_speedup
        batch_trace.append({'start_ms': round(start, 4), 'duration_ms': round(duration, 4), 'request_ids': [item.get('request_id', f"request-{item['_index']}") for item in batch]})
        now += duration; busy += duration
        for item in batch:
            completed.append({
                'request_id': item.get('request_id', f"request-{item['_index']}"),
                'queue_wait_ms': round(start - item['arrival_ms'], 4),
                'ttft_ms': round(start - item['arrival_ms'], 4),
                'e2e_ms': round(now - item['arrival_ms'], 4),
                'output_tokens': item['output_tokens'],
            })
    makespan = now - min(item['arrival_ms'] for item in pending)
    waits = [item['ttft_ms'] for item in completed]
    mean_wait = sum(waits) / len(waits) if waits else 0.0
    spread = (max(waits) - min(waits)) if waits else 0.0
    fairness = max(0.0, 1.0 - spread / max(mean_wait, 1.0))
    total_tokens = sum(item['output_tokens'] for item in completed)
    avg_queue_wait_ms = sum(item['queue_wait_ms'] for item in completed) / len(completed) if completed else 0.0
    return {
        'request_count': len(pending), 'completed_count': len(completed),
        'makespan_ms': round(makespan, 4),
        'throughput_tps': round(total_tokens / (makespan / 1000.0), 4) if makespan else 0.0,
        'utilization': round(busy / makespan, 4) if makespan else 0.0,
        'avg_queue_wait_ms': round(avg_queue_wait_ms, 4),
        'fairness': round(fairness, 4), 'batch_trace': batch_trace, 'requests': completed,
    }

# 辅助函数：汇总调度运行结果
def summarize_serving_scheduler_runs(runs: List[Dict[str, float]]) -> Dict[str, object]:
    """汇总同一请求流和 batch 配置下的调度运行结果。"""
    if not isinstance(runs, list):
        raise TypeError('runs 必须是 list[dict]')
    required = {'name', 'ttft_ms', 'throughput_tps'}
    for index, item in enumerate(runs):
        if not isinstance(item, dict) or not required.issubset(item):
            raise ValueError(f'第 {index} 条 run 必须包含 {sorted(required)}')
    run_count = len(runs)
    avg_throughput_tps = sum(item['throughput_tps'] for item in runs) / run_count if run_count else 0.0
    best = min(runs, key=lambda item: item['ttft_ms']) if runs else None
    return {'run_count': run_count, 'best_latency_run': best.get('name', 'run') if best else None, 'avg_throughput_tps': avg_throughput_tps}

# 辅助函数：比较 baseline 与 candidate
def compare_scheduler_to_baseline(baseline: Dict[str, float], candidate: Dict[str, float]) -> Dict[str, float]:
    """计算 candidate 相对 baseline 的延迟、吞吐、公平性和利用率变化。"""
    required = {'ttft_ms', 'tpot_ms', 'throughput_tps', 'fairness', 'utilization'}
    for name, item in (('baseline', baseline), ('candidate', candidate)):
        if not isinstance(item, dict) or not required.issubset(item):
            raise ValueError(f'{name} 必须包含 {sorted(required)}')
    delta_specs = [
        ('ttft_ms', 'ttft_delta_ms'),
        ('tpot_ms', 'tpot_delta_ms'),
        ('throughput_tps', 'throughput_gain_tps'),
        ('fairness', 'fairness_delta'),
        ('utilization', 'utilization_delta'),
    ]
    comparison = {
        output_key: round(candidate[source_key] - baseline[source_key], 4)
        for source_key, output_key in delta_specs
    }
    return comparison

# TODO 3：输出 serving 调度决策
def recommend_serving_scheduler_run(
    baseline: Dict[str, float], candidate: Dict[str, float], min_throughput_gain: float, min_fairness: float
) -> Dict[str, object]:
    """按延迟、吞吐和公平性门槛给出教学决策。"""
    if min_throughput_gain < 0:
        raise ValueError('min_throughput_gain 不能为负数')
    if not 0 <= min_fairness <= 1:
        raise ValueError('min_fairness 必须位于 [0, 1]')
    comparison = compare_scheduler_to_baseline(baseline, candidate)
    throughput_ok = comparison['throughput_gain_tps'] >= min_throughput_gain
    fairness_ok = candidate['fairness'] >= min_fairness
    latency_ok = comparison['ttft_delta_ms'] < 0 and comparison['tpot_delta_ms'] <= 0
    # TODO 3：只有交互延迟、吞吐增益和公平性同时满足才 accept。
    if latency_ok and throughput_ok and fairness_ok:
        return {'decision': 'accept', 'reason': '延迟、吞吐和公平性都达标，适合进入真实 serving 验证', 'next_action': 'promote_to_serving_rollout'}
    if comparison['throughput_gain_tps'] >= 0 and comparison['utilization_delta'] >= 0:
        return {'decision': 'tune', 'reason': '吞吐和利用率已有改善，但公平性或延迟边界还不够稳', 'next_action': 'refine_queue_rules_or_worker_split'}
    return {'decision': 'reject', 'reason': 'candidate 没有形成可信的 serving 调度收益', 'next_action': 'fallback_to_scheduler_audit'}


### 解析

你可以把这段代码看成一次小型调度实验：先让请求按策略完成，再用结果判断某个策略是否值得继续验证。沿着“请求选择 → batch 服务时间 → 指标对照 → 策略决策”这条链检查即可。

**TODO 1：选择下一批请求**

- FIFO 保留进入队列的顺序；shortest 优先选择 service_ms 较短的请求；priority 优先选择优先级较高的请求。
- batch 不能超过 batch_size；未选中的请求必须留在剩余队列。
- 相同条件下使用 arrival_ms 和 _index 稳定排序，避免结果随输入偶然变化。

**TODO 2：推进 batch 服务时间**

- 同一 batch 的完成时间由批内最长 service_ms 决定，再除以 batch_speedup。
- 不要把 batch 内所有 service_ms 相加；这里用最长服务时间表达请求并行处理后的教学模型。
- 如果 queue wait、makespan 或 throughput 不符合预期，先检查这个时间推进。

**辅助函数：汇总与对照**

- summarize 函数帮助你从同一 workload 的多次运行中找到平均吞吐和最低 TTFT 的运行。
- compare 函数统一使用 candidate - baseline；负的延迟差值和正的吞吐增益分别表示对应方向的改善。

**TODO 3：形成调度决策**

- throughput_ok 检查吞吐增益，fairness_ok 检查 candidate 的公平性，latency_ok 同时检查 TTFT 和 TPOT。
- 三项都满足时可以 accept；只有部分指标改善时先 tune；没有可信收益时 reject。
- 改变门槛后重新运行测试，观察同一 candidate 如何进入不同决策分支。


### Step 5（可选）：GPU 与 backend 实验——真实 serving workload 对照



#### 5.1 环境、输入与固定条件

同一份请求流才能比较调度收益：G0 记录单并发参考，G1 只提高负载以观察服务压力，G2 在相同负载下替换一个可控调度策略。先记录模型、服务端与请求条件；它们会随结果一起写入 JSON。

| 对照组 | 目的 | 保持一致的条件 | 唯一变化 | 需要留下的证据 |
|:---|:---|:---|:---|:---|
| G0 baseline | 建立服务参考值 | 模型、backend、dtype、请求流、生成长度 | concurrency=1 | TTFT、TPOT、吞吐、queue wait |
| G1 load scan | 观察负载压力 | 与 G0 同一模型和请求内容 | concurrency / 到达压力 | P95/P99、utilization、失败状态 |
| G2 strategy compare | 判断调度策略是否值得采用 | 与对照组同一 workload、资源与生成长度 | 一个已确认生效的 scheduler 参数 | 公平性、策略参数与前后差值 |

![GPU/backend serving 实验流程：从预检到 G0、G1、G2 和结果报告](../docs/public/02_PyTorch_Algorithms/70_gpu_serving_benchmark_flow.svg)


In [ ]:
# 5.1：只声明本轮 workload；完整配置在 5.3 统一创建。
from pathlib import Path

WORKLOAD = 'benchmarks/workloads/fixed.jsonl'
if not Path(WORKLOAD).exists():
    print(f'workload 尚未准备：{WORKLOAD}')
else:
    print(f'workload 已找到：{WORKLOAD}')


#### 5.2 环境启动检查（可独立运行）

此检查只读取 PyTorch、GPU、workload 与本地 vLLM 命令是否可用。它不会下载模型或启动服务；换用 GPU、驱动或服务版本后，重新执行并把输出写入实验记录。


In [ ]:
# 5.2 环境预检：只检查运行条件，不启动完整 benchmark。
from pathlib import Path
RUN_REAL_BACKEND = globals().get('RUN_REAL_BACKEND', False)  # 单独运行时默认关闭。
import shutil
import torch
preflight = {
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'workload_exists': Path(WORKLOAD).exists(),
    'vllm_command': shutil.which('vllm') is not None,
    'run_real_backend': RUN_REAL_BACKEND,
}
print('serving preflight:', preflight)


#### 5.3 配置实验条件

下面的单一配置来源定义模型、请求数、并发组和结果文件。G2 选择一种系统级策略：`multi_worker_routing` 比较多个服务 worker 的路由与队列等待；`prefill_decode_split` 比较 Prefill/Decode 分阶段执行与 KV handoff。两类策略都需要服务栈提供真实实现，因此本页通过外部 artifact 接收 G2 结果，而不把单个 vLLM 进程误当作多 worker 或 PD 分离。

| 配置类别 | 当前设置 | 复测时需要确认 |
|:---|:---|:---|
| 模型与 backend | `Qwen2.5-0.5B-Instruct`、vLLM、`auto` dtype | 服务端版本、实际 dtype 与模型路径 |
| workload | 固定 JSONL、5 个请求、64 个生成 token | 请求内容、warmup 与 repeats |
| G0 / G1 | 单并发参考 / 4 并发负载 | 除并发外保持同一条件 |
| G2 路由候选 | 多 worker 路由 | worker 数、路由规则、queue wait 与 P99 |
| G2 分阶段候选 | Prefill/Decode 分离 | PD 拓扑、KV handoff bytes/latency 与策略日志 |


In [ ]:
# 5.3：GPU/backend 实验的唯一配置来源。
from pathlib import Path
import json

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # 同一模型用于所有分组。
BACKEND = 'vllm'  # G0/G1 使用的本地 serving backend。
DTYPE = 'auto'  # 记录 backend 实际选择的 dtype。
RUN_ID = 'smoke'  # 正式复测时替换为可追溯批次标识。
WORKLOAD = 'benchmarks/workloads/fixed.jsonl'  # G0/G1/G2 使用同一请求内容。
NUM_PROMPTS = 5  # 本轮读取的请求数；与 workload 文件一起记录。
MAX_TOKENS = 64  # 每个请求的生成上限。
WARMUP = 1  # 预热轮数。
REPEATS = 1  # 正式复测可增加重复次数。
MIN_FAIRNESS = 0.80  # G2 决策时允许的最低公平性。
RUN_REAL_BACKEND = False  # 默认关闭；确认本地环境后再改为 True。
MANIFEST_PATH = f'benchmarks/results/70_manifest_{RUN_ID}.json'

G0_G1_GROUPS = (
    {'name': 'g0_concurrency1', 'role': 'baseline', 'concurrency': 1},
    {'name': 'g1_concurrency4', 'role': 'load_scan', 'concurrency': 4},
)
# G2 不是单进程 vLLM 的开关。选择已部署的策略，并把其结果 JSON 放到该路径。
G2_MODE = 'prefill_decode_split'  # 也可设为 multi_worker_routing。
G2_RESULT_PATH = f'benchmarks/results/70_g2_{G2_MODE}_{RUN_ID}.json'
USE_EXISTING_G2_ARTIFACT = False  # G2 服务已运行并已生成 JSON 后再改为 True。
G2_PLAN = {
    'name': f'g2_{G2_MODE}',
    'role': 'strategy_candidate',
    'status': 'planned',
    'result_path': G2_RESULT_PATH,
    'required_fields': ['strategy_id', 'strategy_parameters', 'metrics', 'quality', 'evidence'],
    'required_evidence': (
        'worker 数与路由日志、queue wait 与 P99'
        if G2_MODE == 'multi_worker_routing'
        else 'PD 拓扑、KV handoff bytes/latency、Prefill/Decode 时间与策略日志'
    ),
}

try:
    from tools.inference_project_runtime import (
        shared_project_config, start_optional_vllm, stop_optional_vllm,
        run_backend_benchmark,
    )
except ModuleNotFoundError:
    RUN_REAL_BACKEND = False
    def shared_project_config(**kwargs):
        return kwargs

project_config = shared_project_config(
    model=MODEL_ID, backend=BACKEND, dtype=DTYPE, workload=WORKLOAD,
    num_prompts=NUM_PROMPTS, generated_tokens=MAX_TOKENS, warmup=WARMUP,
    repeats=REPEATS, experiment_type='load_sensitivity', groups=G0_G1_GROUPS,
    g2_plan=G2_PLAN,
)
print(project_config)


#### 5.4 执行实验并保存 JSON

先启动一次本地服务并运行 G0/G1。G2 的执行由多 worker router 或 PD 服务栈完成；当它已生成标准结果 JSON 时，勾选读取开关即可把它纳入同一份清单。每个 artifact 都应记录策略参数和证据来源，否则只保留为待复测状态。

| 阶段 | 动作 | 写入的记录 |
|:---|:---|:---|
| 服务启动 | 启动本地 backend 并保存日志路径 | 模型、backend、dtype、启动失败 |
| G0 / G1 | 同一请求流下运行单并发与负载扫描 | 每组 JSON、TTFT、TPOT、吞吐、queue wait |
| G2 artifact | 导入路由或 PD 服务生成的结果 | strategy id、参数、handoff/queue 证据与指标 |
| 清单保存 | 汇总 groups、环境与 artifact 路径 | `70_manifest_{run_id}.json` |
| 失败与复测 | 捕获启动、OOM、timeout 或 artifact 缺字段 | `failure`、`retest_path`、`evidence_level` |


In [ ]:
# 5.4：默认关闭的 G0/G1 backend 入口；可导入已完成的 G2 策略 artifact。
reports = []
failure = None
server = log_path = None

if RUN_REAL_BACKEND:
    try:
        server, log_path, port, selected_dtype, model_path = start_optional_vllm(
            model_id=MODEL_ID, model_source='auto', dtype=DTYPE,
            served_model_name=MODEL_ID,
        )
        for group in G0_G1_GROUPS:
            result_path = f"benchmarks/results/70_{group['name']}_{BACKEND}_{RUN_ID}.json"
            report = run_backend_benchmark(
                project='70', base_url=f'http://127.0.0.1:{port}', model=MODEL_ID,
                label=f"{BACKEND}-{group['name']}", output=result_path,
                workload=WORKLOAD, num_prompts=NUM_PROMPTS, max_tokens=MAX_TOKENS,
                concurrency=group['concurrency'], warmup=WARMUP, repeats=REPEATS,
                backend=BACKEND, dtype=selected_dtype, cache_policy='default',
            )
            reports.append({
                **group, 'report_path': result_path,
                'artifact': {'result_path': result_path, 'artifact_type': 'backend_benchmark_json'},
                'status': report.get('status', 'ok'),
                'quality': report.get('quality', {'status': 'not_evaluated'}),
                'failure': report.get('failure'), 'evidence_level': 'real_backend_smoke',
                'normalized_result': report.get('normalized_result'),
            })
    except Exception as exc:
        failure = {'type': type(exc).__name__, 'message': str(exc)}
    finally:
        if server is not None:
            stop_optional_vllm(server, log_path)

g2_plan = dict(G2_PLAN)
g2_artifact = Path(G2_RESULT_PATH)
if USE_EXISTING_G2_ARTIFACT:
    if not g2_artifact.exists():
        failure = failure or {'type': 'FileNotFoundError', 'message': f'找不到 G2 artifact：{g2_artifact}'}
    else:
        g2_report = json.loads(g2_artifact.read_text(encoding='utf-8'))
        missing = [key for key in G2_PLAN['required_fields'] if key not in g2_report]
        if missing:
            failure = failure or {'type': 'ValueError', 'message': f'G2 artifact 缺少字段：{missing}'}
        else:
            g2_plan['status'] = 'completed'
            reports.append({
                'name': G2_PLAN['name'], 'role': 'strategy_candidate',
                'concurrency': g2_report.get('concurrency'), 'report_path': str(g2_artifact),
                'artifact': {'result_path': str(g2_artifact), 'artifact_type': 'scheduler_strategy_json'},
                'status': g2_report.get('status', 'ok'), 'quality': g2_report.get('quality'),
                'failure': g2_report.get('failure'), 'evidence_level': g2_report.get('evidence', {}).get('level', 'external_backend'),
                'normalized_result': g2_report,
            })

manifest = {
    'project': '70', 'config': project_config, 'groups': reports, 'g2_plan': g2_plan,
    'strategy_metrics': {
        'experiment_type': 'strategy_compare' if g2_plan['status'] == 'completed' else 'load_sensitivity',
        'scheduler_strategy_controlled': g2_plan['status'] == 'completed',
        'queue_wait_required': True,
    },
    'failure': failure, 'evidence_level': 'real_backend_smoke' if reports else 'not_collected',
    'retest_path': MANIFEST_PATH,
}
manifest_file = Path(MANIFEST_PATH)
manifest_file.parent.mkdir(parents=True, exist_ok=True)
manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print({'manifest': str(manifest_file), 'groups': len(reports), 'failure': failure, 'g2': g2_plan['status']})


#### 5.5 读取结果与记录证据

先读取总清单，再打开每组 JSON。结果表只保留用于判断调度的字段：共同配置确认比较是否合法，服务指标解释负载影响，artifact 与失败状态说明证据是否可复现。

| 记录块 | 读取内容 | 用途 |
|:---|:---|:---|
| 实测环境与口径 | model、backend、dtype、workload、生成长度 | 确认 G0/G1 使用相同服务条件 |
| 主线结果 | concurrency、TTFT、TPOT、throughput、queue wait、P99 | 比较单并发与负载变化 |
| 证据状态 | artifact、quality、failure、evidence level | 区分实测、失败与待复测 |


In [ ]:
# 5.5：读取 manifest 与每组 JSON；缺失字段保留为 None，不伪造实测值。
manifest_path = Path(globals().get('MANIFEST_PATH', 'benchmarks/results/70_manifest_smoke.json'))
serving_manifest = None
serving_rows = []

if manifest_path.exists():
    serving_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    for group in serving_manifest.get('groups', []):
        raw = {}
        report_path = Path(group.get('report_path', ''))
        if report_path.exists():
            raw = json.loads(report_path.read_text(encoding='utf-8'))
        normalized = raw.get('normalized_result') or group.get('normalized_result') or raw
        metrics = normalized.get('metrics', raw.get('metrics', {})) if isinstance(normalized, dict) else {}
        config = raw.get('config', normalized.get('config', {})) if isinstance(normalized, dict) else {}
        serving_rows.append({
            'group': group.get('name'), 'role': group.get('role'), 'concurrency': group.get('concurrency'),
            'ttft_ms': metrics.get('ttft_ms'), 'tpot_ms': metrics.get('tpot_ms'),
            'throughput_tps': metrics.get('throughput_tps'), 'queue_wait_ms': metrics.get('queue_wait_ms'),
            'p99_ms': metrics.get('p99_ms'), 'fairness': metrics.get('fairness'),
            'quality': group.get('quality'), 'status': group.get('status'),
            'artifact': group.get('artifact'), 'config': config,
            'strategy_id': normalized.get('strategy_id') if isinstance(normalized, dict) else None,
            'strategy_parameters': normalized.get('strategy_parameters') if isinstance(normalized, dict) else None,
            'evidence': normalized.get('evidence') if isinstance(normalized, dict) else None,
        })
    print({'project': serving_manifest.get('project'), 'rows': serving_rows,
           'failure': serving_manifest.get('failure'), 'g2': serving_manifest.get('g2_plan', {}).get('status')})
else:
    print(f'等待 serving scheduler manifest：{manifest_path}')


#### 5.6 解释结果并形成决策

G0/G1 用来描述当前服务对负载的敏感性；G2 才回答多 worker 路由或 Prefill/Decode 分离是否带来可接受的策略收益。导入 G2 后，先核对策略 id、参数和证据，再同时检查吞吐、TTFT、P99 与公平性，最后把决策写回清单。

| 结果状态 | 决策 | 下一步 |
|:---|:---|:---|
| 缺少 JSON、artifact 字段、失败或质量不通过 | reject / tune | 修复服务、artifact 或质量门槛后复测 |
| 仅有 G0/G1 | tune | 记录负载敏感性，导入一个受控 G2 artifact |
| G2 提高吞吐且未恶化 TTFT/P99、公平性不低于门槛 | accept | 在更多 workload 与重复次数下确认 |
| 指标互相矛盾 | tune | 查看 queue wait、路由或 KV handoff 证据 |


In [ ]:
# 5.6：只在 G2 策略实际受控且证据完整时给出策略 accept。
if serving_manifest is None:
    decision = {'decision': 'pending', 'reason': '尚无 manifest', 'next_action': '执行 5.4 或导入已有 JSON'}
else:
    failure = serving_manifest.get('failure')
    strategy = serving_manifest.get('strategy_metrics', {})
    g2_ready = serving_manifest.get('g2_plan', {}).get('status') == 'completed'
    quality_ok = all((row.get('quality') or {}).get('status') in {'ok', 'passed', 'not_evaluated'} for row in serving_rows)
    if failure:
        decision = {'decision': 'reject', 'reason': 'backend 执行或 artifact 校验失败', 'next_action': '根据 failure 修复后复测'}
    elif not serving_rows:
        decision = {'decision': 'pending', 'reason': '尚无分组 JSON', 'next_action': '执行 5.4'}
    elif not g2_ready or not strategy.get('scheduler_strategy_controlled'):
        decision = {'decision': 'tune', 'reason': '当前仅完成 G0/G1 负载扫描，尚无受控策略对照',
                    'next_action': '导入多 worker 路由或 PD 分离的 G2 artifact'}
    elif not quality_ok:
        decision = {'decision': 'reject', 'reason': '质量门槛未通过', 'next_action': '先修复质量回归'}
    else:
        baseline = next((row for row in serving_rows if row.get('role') == 'baseline'), None)
        candidate = next((row for row in serving_rows if row.get('role') == 'strategy_candidate'), None)
        evidence_ok = candidate and candidate.get('strategy_id') and candidate.get('strategy_parameters') and candidate.get('evidence')
        if baseline is None or candidate is None or not evidence_ok:
            decision = {'decision': 'tune', 'reason': '缺少可核验的 G0/G2 结果或策略证据',
                        'next_action': '补齐 strategy_id、参数和路由/KV handoff 记录'}
        else:
            throughput_ok = (candidate.get('throughput_tps') or 0) >= (baseline.get('throughput_tps') or 0)
            ttft_ok = (candidate.get('ttft_ms') or float('inf')) <= (baseline.get('ttft_ms') or float('inf'))
            p99_ok = (candidate.get('p99_ms') is None or baseline.get('p99_ms') is None
                      or candidate['p99_ms'] <= baseline['p99_ms'])
            fairness_ok = candidate.get('fairness') is not None and candidate['fairness'] >= MIN_FAIRNESS
            if throughput_ok and ttft_ok and p99_ok and fairness_ok:
                decision = {'decision': 'accept', 'reason': 'G2 提升吞吐，且 TTFT/P99 与公平性均达标', 'next_action': '扩大 repeats 与 workload'}
            else:
                decision = {'decision': 'tune', 'reason': 'G2 的吞吐、延迟、P99 或公平性尚未同时达标',
                            'next_action': '检查 queue wait、路由或 KV handoff 证据'}
    serving_manifest['decision'] = decision
    manifest_path.write_text(json.dumps(serving_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(decision)


## 相关阅读

完成 serving 调度 benchmark 后，可继续观察调度器实现、负载扫描和分布式资源组织。下面的项目链接用于连接到后续并行实验，开源实现和开发文档用于对照真实 serving 系统中的调度与 batching 行为。
- [79. Distributed Parallel Benchmark | 分布式并行基准项目](./79_Distributed_Parallel_Benchmark.ipynb)
- [vLLM Scheduler 开源实现](https://github.com/vllm-project/vllm/blob/main/vllm/v1/core/sched/scheduler.py)
- [SGLang Benchmark and Profiling 开发文档](https://github.com/sgl-project/sglang/blob/main/docs_new/docs/developer_guide/benchmark_and_profiling.mdx)
- [SGLang Server Arguments：调度与 batching 配置](https://github.com/sgl-project/sglang/blob/main/docs/advanced_features/server_arguments.md)
